In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 18:49:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')


In [4]:
df_yellow.schema


StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [4]:
output_path = f'data/pq/yellow/2025/11/'
df_yellow \
        .repartition(4) \
        .write.parquet(output_path)
#25MB

In [5]:
df_yellow.registerTempTable('trips_data')

/Applications/XAMPP/xamppfiles/htdocs/de-zoomcamp/.venv/lib/python3.11/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [7]:
df_yellow.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [7]:
spark.sql("""
SELECT
    count(1)
FROM
    trips_data
WHERE tpep_pickup_datetime >= '2025-11-15 00:00:00' and tpep_pickup_datetime< '2025-11-16 00:00:00'
""").show()


+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [12]:
spark.sql("""
SELECT
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0 AS trip_duration_hr
FROM
    trips_data
order by 1 desc
""").show()

[Stage 10:=============================>                            (4 + 4) / 8]

+----------------+
|trip_duration_hr|
+----------------+
|       90.646667|
|       76.948333|
|       76.213889|
|       69.288611|
|       67.080556|
|       63.368333|
|       56.382222|
|       48.147778|
|       47.478333|
|       45.444167|
|       44.043333|
|       43.230556|
|       42.720556|
|       41.614444|
|       41.503333|
|       39.333056|
|       38.074444|
|       37.911111|
|       34.870278|
|       30.533889|
+----------------+
only showing top 20 rows


In [8]:
df_tz_lookup = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [14]:
df_tz_lookup.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [9]:
df_tz_lookup_cleaned = df_tz_lookup \
    .withColumnRenamed('LocationID', 'PULocationID')

In [10]:
df_join = df_yellow.join(df_tz_lookup_cleaned, on=['PULocationID'], how='left')


In [11]:
df_join.registerTempTable('trips_data_mapped')

In [12]:
spark.sql("""
SELECT
    Zone,
    count(1)
FROM
    trips_data_mapped
group by 1
order by 2
""").show()

+--------------------+--------+
|                Zone|count(1)|
+--------------------+--------+
|Governor's Island...|       1|
|Eltingville/Annad...|       1|
|       Arden Heights|       1|
|       Port Richmond|       3|
|       Rikers Island|       4|
|   Rossville/Woodrow|       4|
| Green-Wood Cemetery|       4|
|         Great Kills|       4|
|         Jamaica Bay|       5|
|         Westerleigh|      12|
|        Crotona Park|      14|
|             Oakwood|      14|
|New Dorp/Midland ...|      14|
|       West Brighton|      14|
|       Willets Point|      15|
|Breezy Point/Fort...|      16|
|Saint George/New ...|      17|
|       Broad Channel|      18|
|     Mariners Harbor|      21|
|Heartland Village...|      22|
+--------------------+--------+
only showing top 20 rows
